# 完整建模链路：搜索 → 最终模型 → 评估 → 模型报告

使用真实放款数据，按时间切分训练、选参验证和独立测试。默认单次验证，最终重训单个模型。请按顺序执行以下单元格。

金融指标配置在各阶段复用；训练权重、验证评分权重和报告权重分别显式指定。

## 路径与运行环境

在下方修改输入文件和输出目录。支持环境变量 `HSCREDIT_EXAMPLE_INPUT` / `HSCREDIT_EXAMPLE_OUTPUT`。所有产物写入 `artifacts/`，不改写演示 Notebook。

In [ ]:
from pathlib import Path
import os
import sys
import json
import numpy as np
import pandas as pd
from IPython.display import display

root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / "hscredit").is_dir())
sys.path.insert(0, str(root))
input_path = Path(os.environ.get("HSCREDIT_EXAMPLE_INPUT", root / "examples/hscredit_yyp.xlsx"))
output = Path(os.environ.get("HSCREDIT_EXAMPLE_OUTPUT", root / "artifacts/model_training_chain"))
output.mkdir(parents=True, exist_ok=True)

from hscredit import ModelTuner, RandomForest, get_loss, get_metric

## 1. 读取数据与按时间留出验证/测试集

In [ ]:
data = pd.read_excel(input_path).sort_values("放款时间", kind="stable").reset_index(drop=True)
features = ["衡枢鉴真分老客版", "近六个月非银多头机构数", "青云24"]
target = "FPD"
first, second = int(len(data) * 0.6), int(len(data) * 0.8)
train, valid, test = (part.copy() for part in (data.iloc[:first], data.iloc[first:second], data.iloc[second:]))
# 缺失值处理只从训练数据学习，报告保留金额/时间字段而模型输入仅为指定特征。
medians = train[features].median()
for frame in (train, valid, test):
    frame[features] = frame[features].fillna(medians)
train_input, valid_input, test_input = (frame[[*features, target]] for frame in (train, valid, test))

## 2. 配置模型、金融指标与搜索空间

In [ ]:
# RF 仍按自己的原生目标训练；这里用金融业务指标选择参数，并贯穿评估及报告。
loss = get_loss("cost_sensitive", fn_cost=5, fp_cost=1)
business = loss.metric(kind="business")
auc = get_metric("auc")
base = RandomForest(target=target, n_estimators=24, random_state=42, n_jobs=1)
tuner = ModelTuner(
    base,
    search_space={"max_depth": [2, 3, 4], "min_samples_leaf": [5, 15]},
    metric=business,
    cv=None,
    random_state=42,
    n_jobs=1,
    retention="full",
)

## 3. 单次验证超参数搜索

In [ ]:
best_params = tuner.fit(
    train_input,
    eval_set={"验证集": valid_input},
    n_trials=4,
    show_progress_bar=False,
    # 三种权重含义分开：模型训练、搜索指标和独立报告不隐式共享。
    sample_weight=train["放款金额"].to_numpy(),
    validation_weight=valid["放款金额"].to_numpy(),
)
display(tuner.get_optimization_history())

## 4. 查看搜索过程、取得最终模型并独立评估

In [ ]:
points = tuner.get_optimization_history()
validation_predictions = tuner.get_validation_predictions()
best = tuner.get_best_model()  # 只使用传入训练集重训；不是多个折模型的集成。
effects = best.evaluate(test_input, metrics=[auc, business], sample_weight=test["放款金额"].to_numpy())
display(validation_predictions.head())
display(pd.DataFrame([effects]))

## 5. 生成模型报告并核对指标

In [ ]:
# 传入完整 DataFrame：报告用金额和日期字段做分析，预测仍按训练字段对齐。
report = best.model_report(
    datasets={"训练集": train, "验证集": valid, "独立测试集": test},
    metrics=[auc, business],
    sample_weight="放款金额",
    verbose=False,
    n_jobs=1,
)
report_values = report.get_metrics().set_index("统计项")
for metric in (auc, business):
    np.testing.assert_allclose(report_values.loc[metric.name, "独立测试集"], effects[metric.name])
process = report.get_training_process()
display(report.get_metrics())
display(pd.DataFrame({"过程表": list(process)}))

## 6. 导出完整 Excel 报告

In [ ]:
status = report.to_excel(
    str(output / "完整模型报告.xlsx"),
    amount="放款金额",
    date="放款时间",
    with_plots=True,
    include_sample_records=False,
    mode="strict",
    return_result=True,
)
assert status.complete
display(status.status_table())

## 7. 保存、加载与核对预测

In [ ]:
model_path = best.save(str(output / "最终模型.pkl"))
tuner_path = tuner.save(str(output / "超参数搜索.pkl"))
restored = RandomForest.load(model_path)
np.testing.assert_allclose(restored.predict_proba(test_input), best.predict_proba(test_input))

## 8. 保存可复核的链路结果

In [ ]:
summary = {
    "数据总数": len(data),
    "训练样本": len(train),
    "选参验证样本": len(valid),
    "独立测试样本": len(test),
    "搜索方式": tuner.validation_summary_,
    "搜索点数": len(points),
    "保存的验证预测数": len(validation_predictions),
    "最佳参数": best_params,
    "最终重训样本": best.training_summary_["样本数"],
    "测试效果": effects,
    "模型与报告指标一致": True,
    "模型保存恢复一致": True,
    "报告过程表": list(process),
    "报告完整": status.complete,
    "报告产物": status.artifacts,
    "模型文件": model_path,
    "搜索文件": tuner_path,
}
(output / "链路验收.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2, default=str), encoding="utf-8"
)
result = {"模型": best, "报告": report, "验收": summary}
summary